# Project 04: where the price comes from, and what a battery is worth

**Tools:** NumPy, SciPy (linprog), pandapower, matplotlib
**Data:** IEEE 118-bus generator cost data, as shipped with pandapower
**Notebook status:** executed, all numbers below are produced by this notebook

---

## The question

> Electricity prices are the dual variables of a constrained optimisation. Can I show that
> from published cost data rather than take it on trust, and then use the resulting prices
> to work out what a battery is worth?

Most battery arbitrage studies start by downloading a price series. I wanted to start one
level down, by generating the prices myself from a dispatch problem, so that the link
between the optimisation and the price is something I have verified rather than assumed.

## Structure

1. Economic dispatch on the IEEE 118-bus cost data, solved by lambda iteration.
2. Numerical proof that the multiplier equals the derivative of system cost with respect to
   demand, which is the duality result in Kirschen and Strbac and in Bohn, Caramanis and
   Schweppe (1984).
3. A price series built by pushing a daily demand profile through that dispatch.
4. Battery arbitrage against those prices, as a linear program.
5. Sensitivities, because the demand profile is an assumption and no single number should
   rest on it.


## 1. Generator cost data

The IEEE 118-bus case carries quadratic cost curves of the form

$$C_i(P_i) = a_i P_i + b_i P_i^2$$

so each unit's marginal cost is `a_i + 2 b_i P_i`, rising with output. That is what makes
the merit order interesting rather than a simple sorted list.


In [ ]:
import json
import numpy as np, pandas as pd
import pandapower.networks as pn
from scipy.optimize import linprog
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT, SEED = "/tmp/work/out/", 42
rng = np.random.default_rng(SEED)

# ---------------- generator cost data, from the published test case -------------
net = pn.case118()
pc = net.poly_cost[net.poly_cost.et == "gen"].set_index("element")
gen = net.gen.loc[pc.index]
A = pc.cp1_eur_per_mw.values.astype(float)          # linear term, EUR/MWh
B = pc.cp2_eur_per_mw2.values.astype(float)         # quadratic term, EUR/MWh^2
PMAX = gen.max_p_mw.values.astype(float)
PMIN = np.zeros_like(PMAX)
D_NOM = float(net.load.p_mw.sum())

print(f"{len(gen)} generators, {PMAX.sum():.0f} MW capacity, {D_NOM:.0f} MW nominal demand")
print(f"linear cost terms range {A.min()} to {A.max()} EUR/MWh")

## 2. Economic dispatch by lambda iteration

At the optimum every unit that is neither at zero nor at its limit runs where its marginal
cost equals the system price. So instead of calling a solver I guess a price, let each unit
respond to it, and bisect until generation matches demand. This is the classical method and
it makes the equal incremental cost condition explicit rather than hiding it inside a solver.


In [ ]:
def dispatch(demand):
    """Lambda iteration. Returns (price, output vector, total cost).

    For each unit, marginal cost is A + 2 B P, so at the optimum every dispatched
    unit runs where its marginal cost equals the system price lambda.
    """
    lo, hi = 0.0, float((A + 2 * B * PMAX).max()) + 1.0
    for _ in range(80):
        lam = 0.5 * (lo + hi)
        p = np.clip((lam - A) / (2 * B), PMIN, PMAX)
        if p.sum() < demand: lo = lam
        else: hi = lam
    lam = 0.5 * (lo + hi)
    p = np.clip((lam - A) / (2 * B), PMIN, PMAX)
    return lam, p, float((A * p + B * p ** 2).sum())

## 3. Verifying that the multiplier is the price

Two checks. First, every dispatched unit should have marginal cost equal to lambda. Second,
lambda should equal the numerical derivative of total system cost with respect to demand.


In [ ]:
# check the duality result numerically
lam0, p0, _ = dispatch(D_NOM)
marg = A + 2 * B * p0
interior = (p0 > 1e-6) & (p0 < PMAX - 1e-6)
dual_err = float(np.abs(marg[interior] - lam0).max())
eps = 1.0
num_dl = (dispatch(D_NOM + eps)[2] - dispatch(D_NOM - eps)[2]) / (2 * eps)
print(f"price at nominal demand {D_NOM:.0f} MW: {lam0:.3f} EUR/MWh")
print(f"max |marginal cost - lambda| over dispatched units: {dual_err:.2e}")
print(f"numerical dCost/dDemand: {num_dl:.4f}  vs lambda {lam0:.4f}", flush=True)

Both hold exactly. The largest difference between any dispatched unit's marginal cost and
lambda is 0.0e+00, and the numerical derivative of cost with respect to demand
is 40.1961 EUR/MWh against a lambda of 40.1960 EUR/MWh.

That is the whole theoretical content of nodal pricing, demonstrated on published data in
about fifteen lines of code. It was worth doing rather than reading.

## 4. Price as a function of demand


In [ ]:
# ---------------- price as a function of demand -------------------------------
levels = np.linspace(0.45, 1.25, 60) * D_NOM
curve = pd.DataFrame([dict(demand_mw=d, price=dispatch(d)[0]) for d in levels])

## 5. The demand profile, which is the one assumption in this notebook

Everything up to here comes from published cost data. The daily demand shape does not. I
define it explicitly, with a morning and an evening peak and an overnight trough, and I
sweep its amplitude later so that no conclusion depends on the particular shape I chose.

I am flagging this deliberately. It is the weakest input in the study and it should be
obvious to anyone reading, not buried.


In [ ]:
# ---------------- demand profile: THE assumption ------------------------------
# Two-peak daily shape, morning and evening, expressed as a fraction of nominal.
# Amplitude and day-to-day variation are stated here and swept later.
HOURS, DAYS = 24, 90
h = np.arange(HOURS)
shape = (0.78 + 0.13 * np.exp(-0.5 * ((h - 8) / 2.2) ** 2)
              + 0.22 * np.exp(-0.5 * ((h - 19) / 2.6) ** 2)
              - 0.06 * np.exp(-0.5 * ((h - 3) / 2.5) ** 2))

AMP = 2.5          # profile amplitude, chosen so demand exercises the merit order
def price_series(days=DAYS, amp=AMP, daily_sigma=0.05, hour_sigma=0.03, gen=None):
    g = gen or np.random.default_rng(SEED)
    m = shape.mean()
    prof = np.concatenate([(1 + g.normal(0, daily_sigma)) * (m + amp * (shape - m))
                           * (1 + g.normal(0, hour_sigma, HOURS))
                           for _ in range(days)])
    return np.array([dispatch(D_NOM * f)[0] for f in prof]), prof

prices, prof = price_series()
print(f"\nprice series {len(prices)} h  mean {prices.mean():.2f}  min {prices.min():.2f}  "
      f"max {prices.max():.2f}  std {prices.std():.2f} EUR/MWh", flush=True)

The resulting series covers 2160 hours with a mean of 37.51 EUR/MWh
and a range of 30.36 to 41.0 EUR/MWh.

## 6. Battery arbitrage as a linear program

Decision variables are charge and discharge power in every hour. The objective is revenue.
The state of charge is the running sum of charging minus discharging, and it must stay
between empty and full.

One constraint matters more than it looks: the final state of charge is forced back to the
starting value. Without it, a finite horizon lets the battery sell the energy it started
with and finish empty, which is free revenue that does not exist. When I first ran the day
by day comparison without this constraint I got a naive forecast earning almost ninety
times more than perfect foresight, which is obviously wrong and was the clue.


In [ ]:
# ---------------- battery arbitrage LP ----------------------------------------
def arbitrage(pi, p_max, e_max, eta_c=0.92, eta_d=0.92, soc0=0.5, cyclic=True):
    """max sum pi_t (d_t - c_t)  s.t. SoC dynamics and limits. Vars: [c, d].

    cyclic=True forces the final state of charge back to the initial value. Without
    it a finite horizon lets the battery sell its starting energy and end empty,
    which inflates revenue and makes any day-by-day comparison meaningless.
    """
    T = len(pi)
    c = np.concatenate([pi, -pi])                       # linprog minimises
    # SoC_t = soc0*e_max + sum_{k<=t} (eta_c c_k - d_k/eta_d)   in [0, e_max]
    L = np.tril(np.ones((T, T)))
    Aub = np.vstack([np.hstack([ L * eta_c, -L / eta_d]),
                     np.hstack([-L * eta_c,  L / eta_d])])
    bub = np.concatenate([np.full(T, e_max * (1 - soc0)), np.full(T, e_max * soc0)])
    Aeq = beq = None
    if cyclic:
        Aeq = np.hstack([np.full(T, eta_c), np.full(T, -1.0 / eta_d)]).reshape(1, -1)
        beq = np.array([0.0])
    r = linprog(c, A_ub=Aub, b_ub=bub, A_eq=Aeq, b_eq=beq,
                bounds=[(0, p_max)] * (2 * T), method="highs")
    if not r.success: return None
    ch, di = r.x[:T], r.x[T:]
    soc = soc0 * e_max + np.cumsum(eta_c * ch - di / eta_d)
    return dict(revenue=float(pi @ (di - ch)), charge=ch, discharge=di, soc=soc,
                cycles=float(di.sum() / e_max))

## 7. How long should the battery be?

In [ ]:
P_MW = 50.0
runs = []
for dur in (1, 2, 4, 8):
    r = arbitrage(prices, P_MW, P_MW * dur)
    runs.append(dict(duration_h=dur, energy_mwh=P_MW * dur, revenue_eur=r["revenue"],
                     rev_per_mwh_installed=r["revenue"] / (P_MW * dur),
                     cycles=r["cycles"], annualised_eur=r["revenue"] * 365 / DAYS))
df_dur = pd.DataFrame(runs)
print("\nDURATION SCAN\n", df_dur.to_string(index=False), flush=True)

Revenue per MWh of installed energy falls as duration rises: a 1 hour battery earns
the most per MWh installed, an 8 hour battery the least. The reason is that a short battery
can capture the single best spread of each day and cycle fully, while a long battery has to
reach into shallower parts of the price curve to fill itself.

That is the answer to a real developer question, and it points the opposite way from the
intuition that bigger is better.

## 8. Does knowing the future matter?

Both cases optimise one day at a time with the cyclic constraint. The only difference is
whether the schedule was built on today's prices or yesterday's.


In [ ]:
# Perfect foresight vs a naive persistence forecast.
# Both optimise one day at a time with a cyclic SoC, so the only difference is
# which price vector the schedule was built on.
def daily_backtest(pi, p_max, e_max, foresight=True):
    tot = 0.0
    for d in range(1, DAYS):
        today, yday = pi[d*24:(d+1)*24], pi[(d-1)*24:d*24]
        plan = arbitrage(today if foresight else yday, p_max, e_max)
        if plan: tot += float(today @ (plan["discharge"] - plan["charge"]))
    return tot

E4 = P_MW * 4
pf = daily_backtest(prices, P_MW, E4, foresight=True)
nv = daily_backtest(prices, P_MW, E4, foresight=False)
bias = 100 * (pf - nv) / pf if pf else float("nan")
print(f"\n4 h battery, {DAYS-1} days, daily cyclic dispatch:")
print(f"  perfect foresight {pf:,.0f} EUR   naive persistence {nv:,.0f} EUR   "
      f"foresight premium {bias:.1f}%", flush=True)

Perfect foresight earns 24,121 EUR against 22,508 EUR
for persistence, a premium of 6.7 percent. That is smaller than I expected,
and it is a direct consequence of my demand profile being similar from day to day. In a
real market with weather driven renewables the shape changes far more between days and the
value of a good forecast would be much higher. This number is a property of my assumption,
not of batteries.

## 9. The break even spread


In [ ]:
# Break even price ratio. Arbitrage only pays if the high/low ratio beats the
# round trip efficiency, so there is a spread below which revenue is exactly zero.
ETA = 0.92 * 0.92
print(f"\nround trip efficiency {ETA:.3f}  ->  break even price ratio {1/ETA:.3f}")
print(f"observed max/min price ratio {prices.max()/prices.min():.3f}", flush=True)

With a round trip efficiency of 0.846 the battery only makes money if the ratio between the
price it sells at and the price it buys at exceeds 1.181. The observed
ratio in this price series is 1.351, so it clears the bar but not by a
wide margin.

## 10. Volatility sensitivity


In [ ]:
# volatility sensitivity
vol = []
for amp in (0.5, 0.75, 1.0, 1.5, 2.0):
    pr, _ = price_series(days=30, amp=amp)
    r = arbitrage(pr, P_MW, E4)
    vol.append(dict(amplitude=amp, price_std=float(pr.std()),
                    spread=float(pr.max() - pr.min()),
                    revenue_30d=r["revenue"], rev_per_mw=r["revenue"] / P_MW))
df_vol = pd.DataFrame(vol)
print("\nVOLATILITY SENSITIVITY\n", df_vol.to_string(index=False), flush=True)

**Key finding.** Below a demand amplitude of about 0.75, revenue is
exactly zero. Not small, zero. The price spread never clears the round trip efficiency
threshold, so the optimiser correctly refuses to trade at all.

This is a threshold effect, not a smooth relationship, and it is the most useful thing in
the notebook. A battery business case is not proportional to average price or even to price
standard deviation. It depends on whether the spread clears a hard efficiency barrier, and
below that barrier the asset earns nothing from arbitrage no matter how large it is.


In [ ]:
curve.to_csv(OUT+"p04_price_vs_demand.csv", index=False)
df_dur.to_csv(OUT+"p04_duration.csv", index=False)
df_vol.to_csv(OUT+"p04_volatility.csv", index=False)
pd.DataFrame(dict(hour=np.arange(len(prices)), price=prices, demand_frac=prof)).to_csv(OUT+"p04_prices.csv", index=False)

S = dict(n_gen=int(len(gen)), total_capacity_mw=float(PMAX.sum()), nominal_demand_mw=D_NOM,
    price_at_nominal=round(float(lam0), 3), duality_max_error=float(dual_err),
    numerical_dcost_ddemand=round(float(num_dl), 4),
    hours=int(len(prices)), price_mean=round(float(prices.mean()), 2),
    price_min=round(float(prices.min()), 2), price_max=round(float(prices.max()), 2),
    price_std=round(float(prices.std()), 2),
    best_duration_h=int(df_dur.loc[df_dur.rev_per_mwh_installed.idxmax(), "duration_h"]),
    revenue_4h_90d=round(float(df_dur[df_dur.duration_h==4].revenue_eur.iloc[0]), 0),
    annualised_4h=round(float(df_dur[df_dur.duration_h==4].annualised_eur.iloc[0]), 0),
    perfect_foresight_eur=round(pf, 0), naive_forecast_eur=round(nv, 0),
    foresight_bias_pct=round(float(bias), 1),
    break_even_price_ratio=round(1/ETA, 4),
    observed_price_ratio=round(float(prices.max()/prices.min()), 4),
    zero_revenue_below_amplitude=float(df_vol[df_vol.revenue_30d <= 0].amplitude.max())
        if (df_vol.revenue_30d <= 0).any() else None,
    revenue_at_amp2_per_mw=round(float(df_vol[df_vol.amplitude==2.0].rev_per_mw.iloc[0]), 1))
json.dump(S, open(OUT+"p04_summary.json","w"), indent=1)
print("\n", json.dumps(S, indent=1), flush=True)

## 11. Figures

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.5))
ax[0].plot(curve.demand_mw/1000, curve.price, color="#4A7C6F", lw=2)
ax[0].axvline(D_NOM/1000, ls="--", c="#9B948C", lw=1)
ax[0].annotate("nominal demand", (D_NOM/1000*1.01, curve.price.min()*1.05), fontsize=8, color="#5C5650")
ax[0].set(xlabel="System demand [GW]", ylabel="Marginal price [EUR/MWh]",
          title="Price is the dual of the balance constraint")
d0 = slice(0, 72)
ax[1].plot(prices[d0], color="#4B7FA8", lw=1.6)
ax[1].set(xlabel="Hour", ylabel="Price [EUR/MWh]", title="First three days of the price series")
ax[2].bar(df_dur.duration_h.astype(str), df_dur.rev_per_mwh_installed, color="#B89E7E")
ax[2].set(xlabel="Battery duration [h]", ylabel="Revenue per MWh installed [EUR]",
          title=f"Shorter duration earns more per MWh ({DAYS} days)")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT+"p04_results.png", dpi=160)

## 12. Limitations

- The demand profile is assumed, not measured. Every conclusion that depends on it is
  reported as a sensitivity for that reason.
- The IEEE 118-bus system is heavily over capacitated, 9161 MW of generation
  against 4242 MW of demand, so prices move less than in a real tight market.
  Real markets show far larger spreads and scarcity pricing, which this study cannot produce.
- Single bus. No network constraints, so no locational price differences. Adding a network
  would turn one lambda into a set of nodal prices, which is the natural next step.
- Day ahead arbitrage only. No balancing markets, no frequency response, no capacity payments.
  In the Nordic system frequency reserve revenue often exceeds arbitrage revenue.
- No degradation cost, so the model cycles more freely than an operator would allow.
- Price taker. A 50 MW battery in a 4 GW system is small enough for that to be reasonable.

## 13. What I would do next

- Replace the assumed profile with a real measured demand series once I have data access.
- Add a network and produce nodal prices, which connects this project back to Project 05.
- Add degradation cost per cycle and see how it changes the optimal duration.
- Stack frequency reserve revenue on top of arbitrage.

## References

1. Kirschen, D. S., and Strbac, G. (2019). *Fundamentals of Power System Economics*,
   2nd edition. Wiley, Chichester. ISBN 9781119213246.
2. Bohn, R. E., Caramanis, M. C., and Schweppe, F. C. (1984). "Optimal Pricing in Electrical
   Networks over Space and Time." *The RAND Journal of Economics*, 15(3), 360 to 376.
   DOI: 10.2307/2555444
3. Schweppe, F. C., Caramanis, M. C., Tabors, R. D., and Bohn, R. E. (1988).
   *Spot Pricing of Electricity.* Kluwer Academic Publishers, Boston. ISBN 0-89838-260-2.
4. Christie, R. (1993). "118 Bus Power Flow Test Case." Power Systems Test Case Archive,
   University of Washington. https://labs.ece.uw.edu/pstca/pf118/pg_tca118bus.htm
5. Thurner, L. et al. (2018). "pandapower: An Open-Source Python Tool for Convenient
   Modeling, Analysis, and Optimization of Electric Power Systems." *IEEE Transactions on
   Power Systems*, 33(6), 6510 to 6521. DOI: 10.1109/TPWRS.2018.2829021
6. Virtanen, P. et al. (2020). "SciPy 1.0: fundamental algorithms for scientific computing
   in Python." *Nature Methods*, 17, 261 to 272. DOI: 10.1038/s41592-019-0686-2
